# 1. 누끼 파이프라인 (1_pipeline)

**흐름:** 사진 → **DINO**(글자로 그릇 찾기) → **SAM 2.1**(그릇마다 마스크) → 후처리 → 그릇 후보 목록 → 사용자 선택 → 누끼

* RMBG는 쓰지 않음 (11/13 결정). DINO, SAM 모두 공개 모델이라 **허깅페이스 토큰 필요 없음**
* 파인튜닝 가중치(`weights/sam2_ft.pt`)가 있으면 파인튜닝 SAM 조건이 자동으로 추가되어 비교됨
* 정답: `dataset/gt_inst/train`, `dataset/gt_inst/test` (3번 노트북으로 라벨링). 평가는 **test** 기준

**셀 순서:** 0 설치 → 1 설정 → 2 분류 → 3 모델 → 4 함수 → 5 확인(선택) → 6 전체 실행 → 7 평가 → 8 그림 → 9 최종 출력

# 0. 설치

In [ ]:
!SAM2_BUILD_CUDA=0 pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q pillow-heif

# 1. 경로와 설정
* 사진: `dataset/images/hwang/`, `dataset/images/jeong/` (사진 이름 = 원본 이름, 두 폴더에 같은 이름이 있으면 경고)

In [ ]:
import os, glob, json, time, random, io
import numpy as np
import pandas as pd
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter
from pillow_heif import register_heif_opener
from scipy import ndimage
from google.colab import drive

register_heif_opener()
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AdCanvas_nukki'
IMG_DIR = f'{BASE}/dataset/images'          # 사람별 폴더: images/hwang/, images/jeong/ (train, test 구분 없이)
GT_INST_DIR = f'{BASE}/dataset/gt_inst'     # 그릇별 정답: gt_inst/train/, gt_inst/test/
GT_DIR = f'{BASE}/dataset/gt'               # 흑백 정답 (자동 생성): gt/train/, gt/test/
SPLITS = ('train', 'test')
EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif')


def stem(p):
    return os.path.splitext(os.path.basename(p))[0]


def scan_images():
    """사진 이름(확장자 뺀 원본 이름) → 경로. 사람 폴더(images/hwang 등) 안의 사진만 읽음
    두 폴더에 같은 이름이 있으면 덮어쓰기를 막으려고 목록에서 빼고 경고"""
    found = {}
    for p in sorted(glob.glob(f'{IMG_DIR}/*/*')):
        if p.lower().endswith(EXTS):
            found.setdefault(stem(p), []).append(p)
    dup = {n: ps for n, ps in found.items() if len(ps) > 1}
    loose = [p for p in glob.glob(f'{IMG_DIR}/*') if p.lower().endswith(EXTS)]
    return {n: ps[0] for n, ps in found.items() if len(ps) == 1}, dup, loose


def label_split(n):
    """이 사진의 정답이 어느 폴더(train/test)에 있는지. 없으면 None"""
    return next((s for s in SPLITS if os.path.exists(f'{GT_INST_DIR}/{s}/{n}.png')), None)


IMAGES, DUP, LOOSE = scan_images()
for s in SPLITS:
    os.makedirs(f'{GT_INST_DIR}/{s}', exist_ok=True)
    os.makedirs(f'{GT_DIR}/{s}', exist_ok=True)
if DUP:
    print('경고: 두 폴더에 이름이 같은 사진이 있어서 뺐어요. 한쪽 이름 끝에 _2를 붙여 주세요:')
    for n, ps in DUP.items():
        print('  -', n, ':', [os.path.relpath(p, IMG_DIR) for p in ps])
if LOOSE:
    print(f'참고: images/ 바로 아래 사진 {len(LOOSE)}장은 읽지 않아요. images/hwang/ 또는 images/jeong/으로 옮겨 주세요.')

SEG_DIR = f'{BASE}/seg_pipeline'           # 조건별 결과 마스크, 점수
DINO_DIR = f'{SEG_DIR}/dino_handoff'       # 사진별 DINO 후보 (2번 노트북이 학습 박스로 씀)
W_DIR = f'{BASE}/weights'
FINAL_DIR = f'{BASE}/final'                # VLM팀 전달용 누끼
for d in (SEG_DIR, DINO_DIR, W_DIR, FINAL_DIR):
    os.makedirs(d, exist_ok=True)

def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


device = get_device()

CFG = dict(
    max_side=2048,        # 처리 해상도 (긴 변)
    sam_model='facebook/sam2.1-hiera-small',
    sam_ft_weights=f'{W_DIR}/sam2_ft.pt',
    eval_side=1024,       # 평가 해상도 (긴 변)
    bf_tol=0.0075,        # 경계 점수 허용 거리 (사진 대각선 대비)
    seed=42,
)
CATEGORIES = ['normal', 'white', 'multi', 'hand_tray']

def load_image(p, max_side=2048):
    img = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
    if max_side:
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    return img


def list_images():
    return [IMAGES[n] for n in sorted(IMAGES)]


labels = {s: sorted(stem(p) for p in glob.glob(f'{GT_INST_DIR}/{s}/*.png')) for s in SPLITS}
people = pd.Series([os.path.basename(os.path.dirname(p)) for p in IMAGES.values()]).value_counts().to_dict()
print(f"device: {device} | 사진 {len(IMAGES)}장 {people} | 정답 train {len(labels['train'])}장, test {len(labels['test'])}장")

# 2. 사진 분류 (meta.csv)
* 새 사진은 `normal`로 추가됨. 기준: `LABELING.md` 5번 (우선순위 hand_tray > multi > white > normal)
* 분류는 아래 **2-1번 셀**에서 고침 (드라이브에서 csv를 열면 새 스프레드시트가 생겨서 반영 안 됨)

In [ ]:
META = f'{BASE}/dataset/meta.csv'
meta = pd.read_csv(META) if os.path.exists(META) else pd.DataFrame(columns=['image', 'category'])
meta = meta[meta.image.isin(IMAGES)]                       # 지금 없는 사진은 뺌
new = [n for n in sorted(IMAGES) if n not in set(meta.image)]
if new:
    meta = pd.concat([meta, pd.DataFrame({'image': new, 'category': 'normal'})], ignore_index=True)
    print(f'meta.csv에 새 사진 {len(new)}장 추가 (normal) → 2-1번 셀에서 분류 고치기')
meta.to_csv(META, index=False)
bad = meta[~meta.category.isin(CATEGORIES)]
assert bad.empty, f'category 오타: {bad.to_dict("records")}'
print(meta.category.value_counts().to_dict())

## 2-1. 분류 고치기 (바꿀 사진만 적기)

In [ ]:
fix = {
    # 'IMG_2184': 'multi',
    # 'IMG_1436': 'white',
}
meta = pd.read_csv(META)
sel = meta.image.isin(fix)
meta.loc[sel, 'category'] = meta.loc[sel, 'image'].map(fix)
assert meta.category.isin(CATEGORIES).all(), '철자 확인: normal / white / multi / hand_tray'
meta.to_csv(META, index=False)
print('없는 사진 이름:', [n for n in fix if n not in IMAGES] or '없음')
print(meta.category.value_counts().to_dict())

# 3. 모델 불러오기
* SAM 2.1 원본, 파인튜닝 SAM(가중치가 있을 때만), DINO

In [ ]:
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam_base = SAM2ImagePredictor.from_pretrained(CFG['sam_model'], device=device)

sam_ft = None
if os.path.exists(CFG['sam_ft_weights']):
    sam_ft = SAM2ImagePredictor.from_pretrained(CFG['sam_model'], device=device)
    res = sam_ft.model.load_state_dict(torch.load(CFG['sam_ft_weights'], map_location=device), strict=False)
    print('파인튜닝 가중치 불러옴 | 없는 키', len(res.missing_keys), '| 남는 키', len(res.unexpected_keys))
else:
    print('파인튜닝 가중치 없음 → 원본 SAM만 사용:', CFG['sam_ft_weights'])

from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from torchvision.ops import nms

GD_ID = 'IDEA-Research/grounding-dino-tiny'   # 공개 모델, 허깅페이스 토큰 필요 없음
gd_proc = AutoProcessor.from_pretrained(GD_ID)
gd = AutoModelForZeroShotObjectDetection.from_pretrained(GD_ID).to(device).eval()

DINO_CFG = dict(
    dish_words=['plate', 'bowl', 'dish', 'pot'],            # 그릇 (기본 선택)
    other_words=['cup', 'glass', 'spoon', 'chopsticks'],    # 후보로만 표시 (기본 선택 해제)
    food_words=['food'],                                    # 그릇 없이 놓인 음식
    box_th=0.3,             # 단어별 확신도 중 가장 높은 값의 기준
    nms_iou=0.6,            # 같은 물체에 겹친 박스 정리
    max_area=0.9,           # 사진의 90% 넘는 박스는 버림 (테이블, 배경)
    food_max_area=0.5,      # food 박스는 사진의 50%까지만
)


def box_area(b):
    return (b[2] - b[0]) * (b[3] - b[1])


@torch.no_grad()
def dino_boxes(img):
    """사진 → [{'box', 'label', 'score', 'kind'(dish/other/food), 'dish'(기본 선택)}]"""
    C = DINO_CFG
    words = C['dish_words'] + C['other_words'] + C['food_words']
    kinds = ['dish'] * len(C['dish_words']) + ['other'] * len(C['other_words']) + ['food'] * len(C['food_words'])
    inputs = gd_proc(images=img, text='. '.join(words) + '.', return_tensors='pt').to(device)
    out = gd(**inputs)
    probs = out.logits[0].sigmoid().float().cpu()
    ids = inputs['input_ids'][0].tolist()
    cols = []
    for w in words:
        sub = gd_proc.tokenizer(w, add_special_tokens=False)['input_ids']
        pos = [i + j for i in range(len(ids) - len(sub) + 1) if ids[i:i + len(sub)] == sub for j in range(len(sub))]
        cols.append(probs[:, pos].max(1).values if pos else torch.zeros(len(probs)))
    score, best = torch.stack(cols, 1).max(1)
    W, H = img.size
    cx, cy, bw, bh = out.pred_boxes[0].float().cpu().unbind(-1)
    boxes = torch.stack([(cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H], -1)
    boxes[:, 0::2] = boxes[:, 0::2].clamp(0, W)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(0, H)
    keep = [i for i in range(len(boxes)) if score[i] > C['box_th'] and box_area(boxes[i].tolist()) <
            (C['food_max_area'] if kinds[int(best[i])] == 'food' else C['max_area']) * W * H]
    if not keep:
        return []
    keep = torch.tensor(keep)
    k = keep[nms(boxes[keep], score[keep], C['nms_iou'])].tolist()
    return [{'box': boxes[i].tolist(), 'label': words[int(best[i])], 'score': float(score[i]),
             'kind': kinds[int(best[i])], 'dish': kinds[int(best[i])] != 'other'} for i in k]

# 4. 파이프라인 함수
* `segment_candidates(img)`: 사진 → 그릇 후보 목록 (서비스의 후보 찾기 함수)
* 후처리: 전체 물체 마스크 고르기 → 그릇 모양 정리(음식 채우기) → 그릇 위 음식 중복 제거 → 그릇 없는 음식 정리 → 걸친 수저 합치기
* 원본 SAM은 후보 3개 중 고르고, 파인튜닝 SAM은 학습한 단일 마스크를 그대로 씀

In [ ]:
def mask_box(m):
    ys, xs = np.nonzero(m)
    return [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


def box_mask(shape, box, expand=0.1):
    x0, y0, x1, y1 = box
    ex, ey = expand * (x1 - x0), expand * (y1 - y0)
    out = np.zeros(shape, bool)
    out[max(int(y0 - ey), 0):max(int(np.ceil(y1 + ey)), 0), max(int(x0 - ex), 0):max(int(np.ceil(x1 + ex)), 0)] = True
    return out


def local(fn, m, pad=0):
    """마스크가 있는 영역만 잘라서 계산 (사진 전체로 계산하면 매우 느림)"""
    out = np.zeros_like(m)
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return out
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1])
    return out


def _hull_crop(c):
    pts = cv2.findNonZero(c.astype(np.uint8))
    out = np.zeros(c.shape, np.uint8)
    cv2.fillConvexPoly(out, cv2.convexHull(pts), 1)
    return out.astype(bool)


def hull(m):
    """볼록 외곽 (그릇을 고무줄로 감싼 영역)"""
    return local(_hull_crop, m)


def close_fill(m):
    """틈 메우기 + 구멍 채우기"""
    return local(lambda c: ndimage.binary_fill_holes(ndimage.binary_closing(c, iterations=3)), m, pad=4)


def pick_whole(ms, ss, box, mm_score=0.6, mm_inside=0.9):
    """SAM 후보 마스크 3개(부분 ~ 전체) 중 점수가 충분하고 박스 안에 들어오는 것 중 가장 큰 것 = 전체 물체"""
    ok = [j for j in range(len(ms)) if ms[j].any() and ss[j] >= mm_score * ss.max()]
    bm = box_mask(ms[0].shape, box)
    ok = [j for j in ok if (ms[j] & bm).sum() >= mm_inside * ms[j].sum()]
    return ms[max(ok, key=lambda j: ms[j].sum())] if ok else ms[int(ss.argmax())]


CAND = dict(
    utensil_words=['spoon', 'chopsticks'],   # 그릇에 걸치면 그 그릇에 합침
    utensil_overlap=0.2,      # 수저의 20% 이상이 그릇 외곽 안에 있으면 걸친 것
    part_min=0.1,             # 그릇 마스크 조각 중 가장 큰 조각의 10% 이상만 남김
    solidity=0.9,             # 그릇이 볼록 외곽의 90% 미만이면 외곽으로 채움 (박스 안에서만)
    inside_drop=0.9,          # 다른 그릇 안에 90% 이상 들어간 그릇 후보는 중복으로 제거
    sam_chunk=4,              # SAM을 박스 몇 개씩 나눠 돌릴지 (메모리 절약)
)
COLORS = np.array([(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180),
                   (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212), (0, 128, 128)])


def keep_parts(c, box):
    """가장 큰 조각의 10% 이상이고 박스(조금 넓힌) 안에 절반 이상 있는 조각만 남김"""
    lab, n = ndimage.label(c)
    if n <= 1:
        return c
    idx = np.arange(1, n + 1)
    sizes = ndimage.sum(c, lab, idx)
    inside = ndimage.sum(box_mask(c.shape, box), lab, idx)
    keep = idx[(sizes >= CAND['part_min'] * sizes.max()) & (inside >= 0.5 * sizes)]
    return np.isin(lab, keep if len(keep) else idx[[sizes.argmax()]])


def _crop(m, box, fn, pad=4):
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return m
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out = np.zeros_like(m)
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1], [box[0] - x0, box[1] - y0, box[2] - x0, box[3] - y0])
    return out


def clean_dish(m, box):
    """그릇 마스크 정리: 조각 정리 → 틈 메우기 → 구멍 채우기 → 그래도 많이 비어 있으면 볼록 외곽(박스 안)으로"""
    out = _crop(m, box, lambda c, b: ndimage.binary_fill_holes(ndimage.binary_closing(keep_parts(c, b), iterations=3)))
    if not out.any():
        return out
    h = hull(out) & (box_mask(m.shape, box, 0.05) | out)
    return h if out.sum() < CAND['solidity'] * h.sum() else out


@torch.no_grad()
def sam_masks(predictor, img_np, boxes, kinds=None, multimask=True):
    """박스마다 SAM (박스 밖을 자르지 않고, 합치지 않음). set_image는 사진당 한 번"""
    predictor.set_image(img_np)
    H, W = img_np.shape[:2]
    kinds = kinds or ['dish'] * len(boxes)
    out = []
    for s0 in range(0, len(boxes), CAND['sam_chunk']):
        b = np.array(boxes[s0:s0 + CAND['sam_chunk']], np.float32)
        m, sc, _ = predictor.predict(box=b, multimask_output=multimask)
        m = np.asarray(m).reshape(len(b), -1, H, W) > 0.5
        if not multimask:
            out += list(m[:, 0])
            continue
        sc = np.asarray(sc).reshape(len(b), -1)
        for k in range(len(b)):
            out.append(m[k, int(sc[k].argmax())] if kinds[s0 + k] == 'other' else pick_whole(m[k], sc[k], boxes[s0 + k]))
    return out


def tidy(cands):
    for c in cands:
        if c['dish'] and c['kind'] != 'food':
            c['mask'] = clean_dish(c['mask'], c['box'])
    dishes = sorted([c for c in cands if c['dish'] and c['kind'] != 'food'], key=lambda c: -c['mask'].sum())
    for i, small in enumerate(dishes):
        for big in dishes[:i]:
            if 'dup_of' not in big and (small['mask'] & big['mask']).sum() >= CAND['inside_drop'] * max(small['mask'].sum(), 1):
                small['dish'], small['dup_of'] = False, big['id']
                break
    kept = [d for d in dishes if 'dup_of' not in d]
    dish_union = np.zeros(cands[0]['mask'].shape, bool)
    for d in kept:
        dish_union |= d['mask']
    for f in [c for c in cands if c['kind'] == 'food']:          # 그릇 없이 놓인 음식만 남김
        f['mask'] = close_fill(_crop(f['mask'], f['box'], keep_parts, pad=0))
        a = f['mask'].sum()
        swallows = any((f['mask'] & d['mask']).sum() > 0.3 * d['mask'].sum() for d in kept)
        if a == 0 or (f['mask'] & dish_union).sum() >= 0.5 * a or swallows:
            f['dish'], f['dup_of'] = False, -1
    utensils = [u for u in cands if u['label'] in CAND['utensil_words'] and u['mask'].any()]
    if utensils:   # SAM은 그릇을 딸 때 수저 자리를 파내므로, 그릇의 볼록 외곽과 겹침을 봄
        hulls = {c['id']: hull(c['mask']) for c in cands if c['dish'] and 'dup_of' not in c and c['mask'].any()}
        for u in utensils:
            if not hulls:
                break
            best = max(hulls, key=lambda i: (u['mask'] & hulls[i]).sum())
            if (u['mask'] & hulls[best]).sum() > CAND['utensil_overlap'] * u['mask'].sum():
                d = next(c for c in cands if c['id'] == best)
                d['mask'] = close_fill(d['mask'] | u['mask'])     # 손잡이까지 통째로 포함
                u['merged_into'] = best
    return [c for c in cands if 'dup_of' not in c]


def segment_candidates(img, predictor=None, multimask=None):
    """사진 → (그릇 후보 목록, 정보). 후보 = {'id', 'box', 'label', 'score', 'kind', 'dish'(기본 선택), 'mask'}
    그릇을 하나도 못 찾으면 빈 목록 + 안내 문구 (서비스에서는 사용자가 직접 박스를 그림)"""
    predictor = predictor or sam_base
    if multimask is None:
        multimask = predictor is sam_base
    info, t0 = {}, time.perf_counter()
    found = dino_boxes(img)
    info['t_dino'] = time.perf_counter() - t0
    if not any(f['dish'] for f in found):
        info['message'] = '그릇을 찾지 못했어요. 음식을 직접 표시해 주세요.'
        info['t_total'] = time.perf_counter() - t0
        return [], info
    t1 = time.perf_counter()
    masks = sam_masks(predictor, np.array(img), [f['box'] for f in found], [f['kind'] for f in found], multimask)
    for i, (f, m) in enumerate(zip(found, masks)):
        f.update(id=i, mask=m)
    info['t_sam'] = time.perf_counter() - t1
    t2 = time.perf_counter()
    cands = tidy(found)
    info['t_post'] = time.perf_counter() - t2
    info['t_total'] = time.perf_counter() - t0
    return cands, info


def dish_union(cands):
    """기본 선택(그릇)만 합친 마스크"""
    out = np.zeros(cands[0]['mask'].shape, bool) if cands else None
    for c in cands:
        if c['dish']:
            out |= c['mask']
    return out

# 5. 사진으로 확인해보기 (선택)
* `show_candidates('IMG_2184')` / 여러 장: `show_many([...])` / 전체 사진 격자(분류 확인용): `show_all()`

In [ ]:
def show_candidates(name, predictor=None):
    img = load_image(IMAGES[name], CFG['max_side'])
    base = np.array(img)
    cands, info = segment_candidates(img, predictor)
    times = ' | '.join(f'{k[2:]} {v:.2f}s' for k, v in info.items() if k.startswith('t_'))
    print(f"{name}: 후보 {len(cands)}개 (그릇 {sum(c['dish'] for c in cands)}) | {times}")
    if 'message' in info:
        print(info['message'])
        return cands
    fig, ax = plt.subplots(1, 3, figsize=(18, 7))
    fig.suptitle(f"{name}  |  candidates {len(cands)}, dishes {sum(c['dish'] for c in cands)}  |  {info['t_total']:.2f}s",
                 fontsize=15, fontweight='bold')
    ax[0].imshow(img); ax[0].set_title(f'{name}: candidates  green=dish, orange=other')
    over = base.copy()
    for k, c in enumerate(cands):
        x0, y0, x1, y1 = c['box']
        col = 'lime' if c['dish'] else 'orange'
        ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color=col, lw=2))
        ax[0].text(x0, y0, f"{c['id']}:{c['label']} {c['score']:.2f}", color='k', fontsize=8, backgroundcolor=col)
        if c['dish']:
            over[c['mask']] = (over[c['mask']] * 0.45 + COLORS[k % len(COLORS)] * 0.55).astype(np.uint8)
    ax[1].imshow(over); ax[1].set_title('dish candidates (one color = one dish)')
    ax[2].imshow(dish_union(cands), cmap='gray'); ax[2].set_title('dishes only (default selection)')
    for a in ax:
        a.axis('off')
    plt.tight_layout(); plt.show()
    return cands


def show_many(names, predictor=None):
    for n in names:
        show_candidates(n, predictor)


def show_all(cols=4, rows=3, side=600):
    """전체 사진을 이름, 사람 폴더, 분류, 정답 위치와 함께 격자로 보기"""
    cat = dict(zip(*[pd.read_csv(META)[c] for c in ('image', 'category')])) if os.path.exists(META) else {}
    names = sorted(IMAGES)
    for s in range(0, len(names), cols * rows):
        chunk = names[s:s + cols * rows]
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4.8 * rows), squeeze=False)
        for a in axes.ravel():
            a.axis('off')
        for a, n in zip(axes.ravel(), chunk):
            a.imshow(load_image(IMAGES[n], side))
            who = os.path.basename(os.path.dirname(IMAGES[n]))
            a.set_title(f'{n}\n{who} [{cat.get(n, "?")}] {label_split(n) or "no label"}', fontsize=10)
        fig.suptitle(f'{s + 1}~{s + len(chunk)} / {len(names)}', fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()


cands = show_candidates(sorted(IMAGES)[0])   # 다른 사진: show_candidates('IMG_2184')

# 6. 전체 사진 실행
* `dino_sam`: DINO → 원본 SAM + 후처리 / `dino_sam_ft`: DINO → 파인튜닝 SAM (가중치가 있을 때만)
* 결과: `seg_pipeline/{조건}/{사진}.png` (그릇으로 기본 선택된 후보만 합친 마스크), `dino_handoff/candidates.json`, `seg_time.csv`
* 사진을 추가하면 이 셀만 다시 실행 (세션이 끊겼으면 1, 3, 4번 먼저)

In [ ]:
CONDS = {'dino_sam': sam_base}
if sam_ft is not None:
    CONDS['dino_sam_ft'] = sam_ft
for c in CONDS:
    os.makedirs(f'{SEG_DIR}/{c}', exist_ok=True)

handoff, times = {}, []
for i, name in enumerate(sorted(IMAGES)):
    try:
        img = load_image(IMAGES[name], CFG['max_side'])
    except Exception as e:
        print('건너뜀 (업로드 중이거나 손상된 파일):', name, e)
        continue
    W, H = img.size
    for c, pred in CONDS.items():
        cands, info = segment_candidates(img, pred)
        m = dish_union(cands) if cands else np.zeros((H, W), bool)
        Image.fromarray((m * 255).astype(np.uint8)).save(f'{SEG_DIR}/{c}/{name}.png')
        times.append({'image': name, 'cond': c, 'n_dish': sum(x['dish'] for x in cands), 'sec': round(info['t_total'], 3)})
        if c == 'dino_sam':
            handoff[name] = {'size': [W, H], 'candidates': [{k: x[k] for k in ('box', 'label', 'score', 'kind', 'dish')} for x in cands]}
    print(f"[{i + 1}/{len(IMAGES)}] {name}: 그릇 후보 {times[-len(CONDS)]['n_dish']}개" + ('  ← 그릇 못 찾음' if not handoff.get(name, {}).get('candidates') else ''))

json.dump({'dino': DINO_CFG, 'images': handoff}, open(f'{DINO_DIR}/candidates.json', 'w'), ensure_ascii=False, indent=1)
tdf = pd.DataFrame(times)
tdf.to_csv(f'{SEG_DIR}/seg_time.csv', index=False)
print(tdf.groupby('cond')['sec'].agg(['mean', 'max']).round(3))

# 7. 평가
* 정답(`gt/train`, `gt/test` 흑백)과 비교: IoU(겹침), BF(경계), MAE
* **test가 공식 점수** (학습에 안 쓴 사진). train 점수는 참고용
* 그릇 하나하나의 점수는 2번 노트북 8번 셀

In [ ]:
def to_eval(mask_img, size):
    return np.asarray(mask_img.convert('L').resize(size, Image.BILINEAR), dtype=np.float32) / 255


def boundary_f1(p, g, tol):
    pb = p ^ ndimage.binary_erosion(p)
    gb = g ^ ndimage.binary_erosion(g)
    if not pb.any() and not gb.any():
        return 1.0
    if not pb.any() or not gb.any():
        return 0.0
    prec = (ndimage.distance_transform_edt(~gb)[pb] <= tol).mean()
    rec = (ndimage.distance_transform_edt(~pb)[gb] <= tol).mean()
    return float(2 * prec * rec / (prec + rec + 1e-8))


def metrics(p_soft, g):
    pb = p_soft > 0.5
    union = (pb | g).sum()
    tol = CFG['bf_tol'] * np.hypot(*g.shape)
    return {'IoU': float((pb & g).sum() / union) if union else 1.0, 'BF': boundary_f1(pb, g, tol),
            'MAE': float(np.abs(p_soft - g).mean())}


cat = dict(zip(*[pd.read_csv(META)[c] for c in ('image', 'category')]))
rows = []
for s in SPLITS:
    for gp in sorted(glob.glob(f'{GT_DIR}/{s}/*.png')):
        name = stem(gp)
        gt_img = Image.open(gp)
        k = CFG['eval_side'] / max(gt_img.size)
        size = (round(gt_img.width * k), round(gt_img.height * k))
        g = to_eval(gt_img, size) > 0.5
        for c in CONDS:
            mp = f'{SEG_DIR}/{c}/{name}.png'
            if os.path.exists(mp):
                rows.append({'image': name, 'split': s, 'category': cat.get(name, '?'), 'cond': c,
                             **metrics(to_eval(Image.open(mp), size), g)})

mdf = pd.DataFrame(rows)
if mdf.empty:
    print('정답이 아직 없어요 (3번 노트북으로 라벨링 → 2번 노트북 1-1번 실행)')
else:
    mdf.to_csv(f'{SEG_DIR}/seg_metrics.csv', index=False)
    test = mdf[mdf.split == 'test']
    print(f"정답 train {mdf[mdf.split == 'train'].image.nunique()}장, test {test.image.nunique()}장")
    print('\n=== test (공식 점수) ===')
    print(test.groupby('cond')[['IoU', 'BF', 'MAE']].mean().round(4))
    print('\n=== test 분류별 IoU ===')
    print(test.pivot_table(index='category', columns='cond', values='IoU', aggfunc='mean').round(3))
    print('\n=== train (참고) ===')
    print(mdf[mdf.split == 'train'].groupby('cond')[['IoU', 'BF', 'MAE']].mean().round(4))
    print('\n=== test 사진별 IoU ===')
    print(test.pivot_table(index='image', columns='cond', values='IoU').round(3))

# 8. 샘플 그림 (보고서용)
* test 사진: 원본, 정답, 조건별 결과 → `seg_pipeline/samples_test.png`, 가장 못한 사진 → `samples_worst.png`

In [ ]:
def tint(img_np, m, color=(255, 0, 0)):
    out = img_np.copy()
    out[m] = (out[m] * 0.5 + np.array(color) * 0.5).astype(np.uint8)
    return out


def plot_samples(names, save_name):
    cols = ['image', 'GT'] + list(CONDS)
    fig, axes = plt.subplots(len(names), len(cols), figsize=(3.2 * len(cols), 3.2 * len(names)), squeeze=False)
    for r, name in enumerate(names):
        img = load_image(IMAGES[name], CFG['eval_side'])
        base = np.array(img)
        g = to_eval(Image.open(f'{GT_DIR}/{label_split(name)}/{name}.png'), img.size) > 0.5
        for c, col in enumerate(cols):
            ax = axes[r][c]
            if col == 'image':
                ax.imshow(img); ax.set_title(name, fontsize=9)
            elif col == 'GT':
                ax.imshow(tint(base, g, (0, 200, 0))); ax.set_title('GT', fontsize=9)
            else:
                m = to_eval(Image.open(f'{SEG_DIR}/{col}/{name}.png'), img.size) > 0.5
                iou = mdf[(mdf.image == name) & (mdf.cond == col)].IoU
                ax.imshow(tint(base, m)); ax.set_title(f'{col}\nIoU {iou.iloc[0]:.3f}' if len(iou) else col, fontsize=9)
            ax.axis('off')
    plt.tight_layout(); plt.savefig(f'{SEG_DIR}/{save_name}', dpi=120, bbox_inches='tight'); plt.show()


if not mdf.empty and len(test):
    plot_samples(sorted(test.image.unique())[:12], 'samples_test.png')
    worst_cond = 'dino_sam_ft' if 'dino_sam_ft' in CONDS else 'dino_sam'
    plot_samples(test[test.cond == worst_cond].sort_values('IoU').image.head(6).tolist(), 'samples_worst.png')

# 9. 최종 출력 (VLM팀 전달용)
* 파인튜닝 SAM이 있으면 `dino_sam_ft`, 없으면 `dino_sam` 결과로 `final/`에 저장
* `_image.png` 원본 · `_mask.png` 흑백 마스크 · `_cut.png` 투명 배경 누끼 · `_cut_crop.png` 음식 부분만 · `manifest.json`

In [ ]:
FINAL_COND = 'dino_sam_ft' if 'dino_sam_ft' in CONDS else 'dino_sam'


def make_cutout(img, mask, feather=1):
    a = mask.convert('L')
    if feather:
        a = a.filter(ImageFilter.GaussianBlur(feather))
    out = img.convert('RGBA')
    out.putalpha(a)
    return out


def to_png_bytes(img):
    """FastAPI로 주고받을 때 쓰는 PNG 바이트"""
    buf = io.BytesIO()
    img.save(buf, 'PNG')
    return buf.getvalue()


manifest = {}
for name in sorted(IMAGES):
    mp = f'{SEG_DIR}/{FINAL_COND}/{name}.png'
    if not os.path.exists(mp):
        continue
    img = load_image(IMAGES[name], CFG['max_side'])
    mask = Image.open(mp).convert('L').resize(img.size, Image.NEAREST)
    cut = make_cutout(img, mask)
    bbox = mask.point(lambda v: 255 if v > 127 else 0).getbbox()
    img.save(f'{FINAL_DIR}/{name}_image.png')
    mask.save(f'{FINAL_DIR}/{name}_mask.png')
    cut.save(f'{FINAL_DIR}/{name}_cut.png')
    if bbox:
        cut.crop(bbox).save(f'{FINAL_DIR}/{name}_cut_crop.png')
    manifest[name] = {'image': f'{name}_image.png', 'mask': f'{name}_mask.png', 'cut': f'{name}_cut.png',
                      'cut_crop': f'{name}_cut_crop.png' if bbox else None, 'size': list(img.size),
                      'food_bbox': list(bbox) if bbox else None, 'seg_cond': FINAL_COND, 'no_food': bbox is None}
json.dump(manifest, open(f'{FINAL_DIR}/manifest.json', 'w'), ensure_ascii=False, indent=1)
print(f'{len(manifest)}장 저장 → {FINAL_DIR} (조건: {FINAL_COND}) | 그릇 못 찾은 사진:', [n for n, v in manifest.items() if v['no_food']])